# Tutorial 06 - FOOOF/Specparam on Selected Segments

This notebook computes PSDs for the selected 1-minute segments and fits spectral parameterization models to separate aperiodic and periodic components.

Clinical EEG is messy. Treat model parameters as descriptive features, not direct biological truths.

In [ ]:
import numpy as np
import pandas as pd
import scipy.io as sio
from scipy.signal import welch
import matplotlib.pyplot as plt
from pathlib import Path

# Try specparam first, then fallback to fooof package naming
model_backend = None
try:
    from specparam import SpectralModel
    model_backend = 'specparam'
except Exception:
    try:
        from fooof import FOOOF as SpectralModel
        model_backend = 'fooof'
    except Exception:
        model_backend = None

print('Model backend:', model_backend)
if model_backend is None:
    print('Install one package before running fits: pip install specparam   OR   pip install fooof')

mat_path = Path('..') / 'data' / 'MGH4J_sid001_1d8_20130718_075948.mat'
out_dir = Path('..') / 'outputs'
sel_path = out_dir / 'selected_one_minute_segments.csv'

mat = sio.loadmat(str(mat_path), squeeze_me=True, struct_as_record=False)
bipolar = np.asarray(mat['EEG_segs_bipolar'])
channel_names = [str(x).strip() for x in np.asarray(mat['channel_names']).ravel()]
Fs = float(mat['Fs'])

if not sel_path.exists():
    raise FileNotFoundError(f'Missing selection file: {sel_path}. Run Tutorial 05 first.')

selected_df = pd.read_csv(sel_path)
print('Selected 1-minute windows:', len(selected_df))
selected_df.head()

## 1) Define PSD and model parameters explicitly

You should tune these parameters for your analysis and document any changes.

In [ ]:
# Welch settings
welch_nperseg = 800
welch_noverlap = 400
welch_window = 'hann'

# Model fitting settings
freq_range = [1, 40]
peak_width_limits = [1, 12]
max_n_peaks = 6
min_peak_height = 0.05
aperiodic_mode = 'fixed'

print('Welch frequency resolution: {:.3f} Hz'.format(Fs / welch_nperseg))
print('Fitting range:', freq_range, 'Hz')

## 2) Compute PSDs for each selected segment and channel

In [ ]:
rows = []
stored_psd = {}

for _, seg_row in selected_df.iterrows():
    seg_rank = int(seg_row['selection_rank'])
    start = int(seg_row['global_start_sample'])

    seg_start = start // 1000
    n_target = int(60 * Fs)
    seg_end = (start + n_target - 1) // 1000

    for ch_idx, ch_name in enumerate(channel_names):
        part = bipolar[seg_start:seg_end + 1, ch_idx, :]
        sig = np.concatenate([part[i] for i in range(part.shape[0])])
        offset = start - seg_start * 1000
        sig = sig[offset:offset + n_target]

        f, p = welch(sig, fs=Fs, window=welch_window, nperseg=welch_nperseg, noverlap=welch_noverlap, detrend='constant')

        rows.append({
            'selection_rank': seg_rank,
            'channel_index': ch_idx,
            'channel': ch_name,
            'freq': f,
            'psd': p
        })
        stored_psd[(seg_rank, ch_idx)] = (f, p)

print('Computed PSDs for entries:', len(rows))

## 3) Fit specparam/FOOOF models and collect features

We keep model calls explicit inside the loop so students can inspect every step.

In [ ]:
if model_backend is None:
    fit_df = pd.DataFrame()
    print('Skipping fits because specparam/fooof is not installed.')
else:
    fit_rows = []
    for item in rows:
        seg_rank = item['selection_rank']
        ch_idx = item['channel_index']
        ch_name = item['channel']
        f = item['freq']
        p = item['psd']

        fm = SpectralModel(
            peak_width_limits=peak_width_limits,
            max_n_peaks=max_n_peaks,
            min_peak_height=min_peak_height,
            aperiodic_mode=aperiodic_mode,
            verbose=False
        )

        fm.fit(f, p, freq_range)

        # Accessors differ slightly across versions, so use robust fallbacks.
        ap = getattr(fm, 'aperiodic_params_', [np.nan, np.nan])
        peaks = getattr(fm, 'peak_params_', np.empty((0, 3)))
        r2 = getattr(fm, 'r_squared_', np.nan)
        err = getattr(fm, 'error_', np.nan)

        if peaks is None or len(peaks) == 0:
            top_cf = np.nan
            top_pw = np.nan
            top_bw = np.nan
            n_peaks = 0
        else:
            peaks = np.asarray(peaks)
            idx = int(np.argmax(peaks[:, 1]))
            top_cf = float(peaks[idx, 0])
            top_pw = float(peaks[idx, 1])
            top_bw = float(peaks[idx, 2])
            n_peaks = int(peaks.shape[0])

        fit_rows.append({
            'selection_rank': seg_rank,
            'channel_index': ch_idx,
            'channel': ch_name,
            'aperiodic_offset': float(ap[0]) if len(ap) > 0 else np.nan,
            'aperiodic_exponent': float(ap[1]) if len(ap) > 1 else np.nan,
            'n_peaks': n_peaks,
            'top_peak_cf_hz': top_cf,
            'top_peak_power': top_pw,
            'top_peak_bw_hz': top_bw,
            'fit_r2': float(r2),
            'fit_error': float(err)
        })

    fit_df = pd.DataFrame(fit_rows)

fit_df.head()

In [ ]:
if model_backend is not None and len(rows) > 0:
    # Plot one representative fit for visual inspection
    rep = rows[0]
    fm = SpectralModel(
        peak_width_limits=peak_width_limits,
        max_n_peaks=max_n_peaks,
        min_peak_height=min_peak_height,
        aperiodic_mode=aperiodic_mode,
        verbose=False
    )
    fm.fit(rep['freq'], rep['psd'], freq_range)

    try:
        fm.plot()
        plt.title(f"Representative fit - selection {rep['selection_rank']}, channel {rep['channel']}")
        plt.show()
    except Exception as e:
        print('Fit plotting unavailable in this package version:', e)

if not fit_df.empty:
    print(fit_df[['fit_r2', 'fit_error', 'n_peaks']].describe())

In [ ]:
features_path = out_dir / 'fooof_features_segment_channel.csv'
quality_path = out_dir / 'fooof_fit_quality_summary.csv'

if not fit_df.empty:
    fit_df.to_csv(features_path, index=False)
    quality_summary = fit_df[['fit_r2', 'fit_error', 'n_peaks']].describe().reset_index()
    quality_summary.to_csv(quality_path, index=False)
    print('Saved:', features_path.resolve())
    print('Saved:', quality_path.resolve())
else:
    print('No feature table saved because no fits were run.')

## Interpretation cautions

- Aperiodic and periodic parameters are useful descriptive summaries of spectral shape.
- They are not direct biological ground truth.
- Always pair model parameters with fit quality checks and raw PSD inspection.

Tutorial 07 will aggregate these outputs into tidy segment-level and patient-level summaries.